#### Step1 Search for primate orthologs of each gene from the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) 

In [ ]:
orth = {}
for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    orth.update({locus: record['IdList']})

#### 

In [ ]:
import time
import xmltodict as x2d
from Bio import Entrez

def as_list(x):
    """
    xmltodict sometimes returns a dictionary if there is only one item,
    and a list if there are multiple items.
    This function makes everything easier to loop through.
    """
    if x is None:
        return []
    if isinstance(x, list):
        return x
    return [x]


def extract_protein_accessions(obj):
    """
    Recursively search the NCBI Gene XML dictionary and collect protein accessions.
    Returns accessions like XP_077799762.1 or NP_000148.2.
    """
    proteins = set()

    if isinstance(obj, dict):
        acc = obj.get("Gene-commentary_accession")
        ver = obj.get("Gene-commentary_version")

        if acc is not None:
            # Protein accessions usually start with NP_, XP_, or YP_
            if acc.startswith(("NP_", "XP_", "YP_")):
                if ver is not None:
                    proteins.add(f"{acc}.{ver}")
                else:
                    proteins.add(acc)

        for value in obj.values():
            proteins.update(extract_protein_accessions(value))

    elif isinstance(obj, list):
        for item in obj:
            proteins.update(extract_protein_accessions(item))

    return proteins


def fetch_gene_record(gene_id):
    """
    Fetch one Entrez Gene XML record and parse it with xmltodict.
    """
    handle = Entrez.efetch(db="gene", id=gene_id, rettype="xml", retmode="text")
    record = x2d.parse(handle.read().decode("utf-8"))
    handle.close()
    return record


# obtain primate Orthologs per gene
result = {}

for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    search_record = Entrez.read(handle)
    handle.close()

    gene_ids = search_record["IdList"]

    print(f"{locus}: {search_record['Count']} -> {gene_ids[:4]}")

    result[locus] = {}

    for gene_id in gene_ids:
        try:
            gene_record = fetch_gene_record(gene_id)

            protein_ids = sorted(extract_protein_accessions(gene_record))

            result[locus][gene_id] = protein_ids

            print(f"  {gene_id}: {len(protein_ids)} proteins")

            # Be polite to NCBI servers
            time.sleep(0.35)

        except Exception as e:
            print(f"  Error with {locus} / {gene_id}: {e}")
            result[locus][gene_id] = []

ABCD1: 35 -> ['215', '6367', '696794', '465923']
  215: 4 proteins
  6367: 5 proteins
  696794: 3 proteins
  465923: 1 proteins
  102120996: 5 proteins
  100409370: 1 proteins
  138379168: 3 proteins
  129475142: 4 proteins
  129023948: 2 proteins
  128576678: 2 proteins
  126945380: 1 proteins
  123628458: 2 proteins
  117078183: 1 proteins
  116812184: 2 proteins
  116541212: 1 proteins
  112615076: 2 proteins
  108535081: 2 proteins
  108294745: 1 proteins
  105867309: 2 proteins
  105824530: 2 proteins
  105707094: 1 proteins
  105598745: 1 proteins
  105555352: 2 proteins
  105521622: 2 proteins
  104661178: 2 proteins
  103251037: 1 proteins
  103232990: 3 proteins
  101125363: 1 proteins
  101029873: 1 proteins
  101015478: 1 proteins
  100974677: 1 proteins
  100947429: 3 proteins
  100586126: 1 proteins
  100441172: 2 proteins
  105467327: 2 proteins
ALPL: 35 -> ['249', '717809', '456600', '100401643']
  249: 8 proteins
  717809: 9 proteins
  456600: 6 proteins
  100401643: 10

#### 1. Get all protein IDs for each gene from your result dictionary
#### 2. Fetch protein records from NCBI
#### 3. Store nested dictionary of actual sequences
#### 4. Collect unique sequences, human first then non-human
#### 5. Save FASTA file for each gene in /Fasta folder


In [ ]:
from pathlib import Path
from datetime import datetime
import time
import pandas as pd
from Bio import Entrez, SeqIO

# Make sure fasta folder exists
#Path("fasta").mkdir(exist_ok=True)
mkdir fasta

def flatten_protein_ids(variant_dict):
    """
    Convert:
        {"variant1": [ids], "variant2": [ids]}
    into one unique list of protein IDs for that gene.
    """
    seq_ids = []
    seen = set()

    for variant_id, protein_ids in variant_dict.items():
        for pid in protein_ids:
            if pid not in seen:
                seq_ids.append(pid)
                seen.add(pid)

    return seq_ids


def fetch_protein_records(seq_ids, chunk_size=200):
    """
    Fetch GenBank protein records from NCBI.
    Uses chunks so the request does not become too large.
    """
    records = []

    for start in range(0, len(seq_ids), chunk_size):
        chunk = seq_ids[start:start + chunk_size]

        with Entrez.efetch(
            db="protein",
            rettype="gb",
            retmode="text",
            id=",".join(chunk)
        ) as handle:
            records.extend(list(SeqIO.parse(handle, "gb")))

        time.sleep(0.35)

    return records


def collect_unique_sequences_human_first(records):
    """
    Keep unique protein sequences.
    First collect unique Homo sapiens sequences,
    then collect unique non-human sequences.
    """
    seq_to_index = {}
    unique_records = []
    excluded_records = []

    inc = 0
    exc = 0

    # 1. Collect unique human sequences first
    for seq_record in records:
        sp = get_species(seq_record)

        if sp == "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(
                    f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***"
                )
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(
                    f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                )
                inc += 1

    # 2. Collect other unique non-human sequences
    for seq_record in records:
        sp = get_species(seq_record)

        if sp != "Homo sapiens":
            seq = str(seq_record.seq)

            if seq in seq_to_index:
                excluded_records.append(seq_record)
                print(
                    f"\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                    f"\t*** same as sequence {seq_to_index[seq]} excluded ***"
                )
                exc += 1
            else:
                seq_to_index[seq] = inc
                unique_records.append(seq_record)
                print(
                    f"{inc}:\t{seq_record.id}\t({len(seq)} aa)\t{seq_record.description}"
                )
                inc += 1

    return unique_records, excluded_records, inc, exc


def make_sequence_dict(variant_dict, records):
    """
    Preserve your original nested structure, but replace each protein ID
    with its actual protein sequence.

    Output:
        {
            "variant1": {
                "XP_...": "MEEPQSD...",
                "NP_...": "MEEPQSD..."
            }
        }
    """
    seq_by_id = {seq_record.id: str(seq_record.seq) for seq_record in records}

    sequence_dict = {}

    for variant_id, protein_ids in variant_dict.items():
        sequence_dict[variant_id] = {}

        for pid in protein_ids:
            sequence_dict[variant_id][pid] = seq_by_id.get(pid, None)

    return sequence_dict

In [ ]:
protein_sequences = {}
unique_records_by_gene = {}
excluded_records_by_gene = {}

summary = []

for gene, variant_dict in result.items():

    print("\n" + "=" * 80)
    print(datetime.now())
    print(f"{gene} orthologs")

    # Get all protein IDs for this gene from your result dictionary
    seq_ids = flatten_protein_ids(variant_dict)

    print(f"{gene}: {len(seq_ids)} protein IDs")
    print(f"Sequence IDs: {seq_ids[:10]}{' ...' if len(seq_ids) > 10 else ''}")

    if len(seq_ids) == 0:
        protein_sequences[gene] = {}
        unique_records_by_gene[gene] = []
        excluded_records_by_gene[gene] = []

        summary.append({
            "gene": gene,
            "n_protein_ids": 0,
            "n_records_fetched": 0,
            "n_unique_sequences": 0,
            "n_excluded_duplicates": 0,
            "fasta_file": None
        })

        continue

    # Fetch protein records from NCBI
    records = fetch_protein_records(seq_ids)

    print(f"{gene}: {len(records)} protein records fetched")

    # Store nested dictionary of actual sequences
    protein_sequences[gene] = make_sequence_dict(variant_dict, records)

    # Collect unique sequences, human first
    print(f"\n{gene} orthologs: unique primate sequences\n")

    unique_records, excluded_records, inc, exc = collect_unique_sequences_human_first(records)

    unique_records_by_gene[gene] = unique_records
    excluded_records_by_gene[gene] = excluded_records

    # Save FASTA file for that gene
    fasta_path = f"fasta/{gene}.fasta"

    with open(fasta_path, "w") as output:
        SeqIO.write(unique_records, output, "fasta")

    print(f"\nTotal: {inc} unique sequences, {exc} excluded")
    print(f"{fasta_path} saved!")

    summary.append({
        "gene": gene,
        "n_protein_ids": len(seq_ids),
        "n_records_fetched": len(records),
        "n_unique_sequences": inc,
        "n_excluded_duplicates": exc,
        "fasta_file": fasta_path
    })

summary_df = pd.DataFrame(summary)
summary_df